# Main Analysis Notebook

This notebook combines treatment panels across sessions and includes the tournament-effect robustness check.

In [ ]:
from analysis.load import load_session
from analysis.metrics import treatment_panel
from analysis.robustness import tournament_effect_robustness
import pandas as pd

In [ ]:
# Update this list as new sessions are collected.
session_ids = [1]

panels = []
for sid in session_ids:
    f = load_session(sid)
    p = treatment_panel(f)
    p['session_id'] = sid
    panels.append(p)

all_panel = pd.concat(panels, ignore_index=True) if panels else pd.DataFrame()
all_panel.head()

In [ ]:
stage_results = all_panel.groupby(['stage'], as_index=False).agg(
    mean_abs_deviation=('abs_deviation', 'mean'),
    median_abs_deviation=('abs_deviation', 'median'),
    mean_turnover=('turnover', 'mean'),
    n_rounds=('round_id', 'count')
)
stage_results

In [ ]:
scenario_stage = all_panel.groupby(['scenario_id', 'stage'], as_index=False).agg(
    mean_abs_deviation=('abs_deviation', 'mean'),
    mean_turnover=('turnover', 'mean'),
    n=('round_id', 'count')
)
scenario_stage.sort_values(['stage', 'scenario_id'])

## Tournament-Effect Robustness Check

Runs the Phase-7 check: regress round-level price-path-deviation on cumulative interim balance position.

In [ ]:
robustness_tables = []
for sid in session_ids:
    frames = load_session(sid)
    res = tournament_effect_robustness(frames)
    tbl = res.regression_table.copy()
    tbl['session_id'] = sid
    robustness_tables.append(tbl)

robustness_df = pd.concat(robustness_tables, ignore_index=True) if robustness_tables else pd.DataFrame()
robustness_df

## Next steps

1. Expand `session_ids` as data collection proceeds.
2. Add hypothesis-specific models (e.g., stage contrasts, robustness checks).
3. Export publication tables to `analysis/output/` via scripts.